In [0]:
from pyspark.sql.functions import *

In [0]:
crm_product_df = spark.table("bronze.crm_products")

In [0]:
display(crm_product_df.limit(10))

prd_id,prd_key,prd_nm,prd_cost,prd_line,prd_start_dt,prd_end_dt
210,CO-RF-FR-R92B-58,HL Road Frame - Black- 58,null,R,2003-07-01,null
211,CO-RF-FR-R92R-58,HL Road Frame - Red- 58,null,R,2003-07-01,null
212,AC-HE-HL-U509-R,Sport-100 Helmet- Red,12,S,2011-07-01,2007-12-28
213,AC-HE-HL-U509-R,Sport-100 Helmet- Red,14,S,2012-07-01,2008-12-27
214,AC-HE-HL-U509-R,Sport-100 Helmet- Red,13,S,2013-07-01,null
215,AC-HE-HL-U509,Sport-100 Helmet- Black,12,S,2011-07-01,2007-12-28
216,AC-HE-HL-U509,Sport-100 Helmet- Black,14,S,2012-07-01,2008-12-27
217,AC-HE-HL-U509,Sport-100 Helmet- Black,13,S,2013-07-01,null
218,CL-SO-SO-B909-M,Mountain Bike Socks- M,3,M,2011-07-01,2007-12-28
219,CL-SO-SO-B909-L,Mountain Bike Socks- L,3,M,2011-07-01,2007-12-28


In [0]:
product_df = (
    crm_product_df
    .withColumnRenamed("prd_id", "product_id")
    .withColumnRenamed("prd_key", "product_key")
    .withColumnRenamed("prd_nm", "product_name")
    .withColumnRenamed("prd_cost", "product_cost")
    .withColumnRenamed("prd_line", "product_line")
    .withColumnRenamed("prd_start_dt", "start_date")
    .withColumnRenamed("prd_end_dt", "end_date")
)

In [0]:
product_df = (
    product_df
    .withColumn(
        "product_name",
        initcap(trim(col("product_name")))
    )
)

In [0]:
product_df = (
    product_df
    .withColumn(
        "product_line",
        when(col("product_line") == "M", "Mountain")
        .when(col("product_line") == "R", "Road")
        .when(col("product_line") == "S", "Other Sales")
        .when(col("product_line") == "T", "Touring")
        .otherwise("Unknown")
    )
)

In [0]:
product_df = (
    product_df

    .withColumn(
        "product_cost",
        col("product_cost").cast("double")
    )
)

In [0]:
product_df = (
    product_df
    .withColumn(
        "start_date",
        to_date(col("start_date"))
    )
    .withColumn(
        "end_date",
        to_date(col("end_date"))
    )
)

In [0]:
product_df = product_df.dropDuplicates(["product_id"])

In [0]:
product_df = product_df.orderBy("product_id")

In [0]:
print("PRODUCT TRANSFORMATION SUMMARY")
print(f"Total Products : {product_df.count():,}")
print(f"Total Columns  : {len(product_df.columns)}")
print("\nSchema")
product_df.printSchema()

PRODUCT TRANSFORMATION SUMMARY
Total Products : 397
Total Columns  : 7

Schema
root
 |-- product_id: integer (nullable = true)
 |-- product_key: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_cost: double (nullable = true)
 |-- product_line: string (nullable = false)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)



In [0]:
display(product_df.limit(10))

product_id,product_key,product_name,product_cost,product_line,start_date,end_date
210,CO-RF-FR-R92B-58,Hl Road Frame - Black- 58,null,Unknown,2003-07-01,null
211,CO-RF-FR-R92R-58,Hl Road Frame - Red- 58,null,Unknown,2003-07-01,null
212,AC-HE-HL-U509-R,Sport-100 Helmet- Red,12.0,Unknown,2011-07-01,2007-12-28
213,AC-HE-HL-U509-R,Sport-100 Helmet- Red,14.0,Unknown,2012-07-01,2008-12-27
214,AC-HE-HL-U509-R,Sport-100 Helmet- Red,13.0,Unknown,2013-07-01,null
215,AC-HE-HL-U509,Sport-100 Helmet- Black,12.0,Unknown,2011-07-01,2007-12-28
216,AC-HE-HL-U509,Sport-100 Helmet- Black,14.0,Unknown,2012-07-01,2008-12-27
217,AC-HE-HL-U509,Sport-100 Helmet- Black,13.0,Unknown,2013-07-01,null
218,CL-SO-SO-B909-M,Mountain Bike Socks- M,3.0,Unknown,2011-07-01,2007-12-28
219,CL-SO-SO-B909-L,Mountain Bike Socks- L,3.0,Unknown,2011-07-01,2007-12-28


In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

DataFrame[]

In [0]:

product_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("silver.products")

In [0]:
spark.sql("SHOW TABLES IN silver").show()

display(spark.table("silver.products"))

+--------+---------+-----------+
|database|tableName|isTemporary|
+--------+---------+-----------+
|  silver|customers|      false|
|  silver| products|      false|
+--------+---------+-----------+



product_id,product_key,product_name,product_cost,product_line,category,subcategory,maintenance,start_date,end_date
210,CO-RF-FR-R92B-58,Hl Road Frame - Black- 58,null,Unknown,null,null,null,2003-07-01,null
211,CO-RF-FR-R92R-58,Hl Road Frame - Red- 58,null,Unknown,null,null,null,2003-07-01,null
212,AC-HE-HL-U509-R,Sport-100 Helmet- Red,12.0,Unknown,null,null,null,2011-07-01,2007-12-28
213,AC-HE-HL-U509-R,Sport-100 Helmet- Red,14.0,Unknown,null,null,null,2012-07-01,2008-12-27
214,AC-HE-HL-U509-R,Sport-100 Helmet- Red,13.0,Unknown,null,null,null,2013-07-01,null
215,AC-HE-HL-U509,Sport-100 Helmet- Black,12.0,Unknown,null,null,null,2011-07-01,2007-12-28
216,AC-HE-HL-U509,Sport-100 Helmet- Black,14.0,Unknown,null,null,null,2012-07-01,2008-12-27
217,AC-HE-HL-U509,Sport-100 Helmet- Black,13.0,Unknown,null,null,null,2013-07-01,null
218,CL-SO-SO-B909-M,Mountain Bike Socks- M,3.0,Unknown,null,null,null,2011-07-01,2007-12-28
219,CL-SO-SO-B909-L,Mountain Bike Socks- L,3.0,Unknown,null,null,null,2011-07-01,2007-12-28
